In [ ]:
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import HistGradientBoostingRegressor
from scipy.stats import norm

In [ ]:
df=pd.read_excel("nba_dataset.xlsx")

## Data Exploration

### Check duplicates and missing values

In [ ]:
print(df.shape)
duplicates_rows=df.duplicated().sum()
print("duplicates:", duplicates_rows)

if "is_home" in df.columns:
    print("is_home unique values:", sorted(df["is_home"].dropna().unique().tolist()))
    home_balance = df.groupby("game_id")["is_home"].agg(["sum", "count"])
    bad_home_balance = home_balance[
        ~((home_balance["count"] == 2) & (home_balance["sum"] == 1))
    ]
    print("\nGames without exactly one home team:", len(bad_home_balance))
#missing values
missing = (
    df.isna()
      .sum()
      .to_frame("missing_count")
      .assign(missing_pct=lambda x: x["missing_count"] / len(df))
      .sort_values(["missing_pct", "missing_count"], ascending=False)
)

print(missing.head(50))

missing_nonzero = missing[missing["missing_count"] > 0]
print("\nColumns with missing values:")
print(missing_nonzero)



(14746, 125)
duplicates: 0
is_home unique values: [0, 1]

Games without exactly one home team: 0
                                   missing_count  missing_pct
Post_attempts                                 62     0.004205
Post_makes                                    62     0.004205
Post_rim_attempts                             62     0.004205
Post_three_attempts                           62     0.004205
Pick'n'roll_roller_attempts                   58     0.003933
Pick'n'roll_roller_makes                      58     0.003933
Pick'n'roll_roller_rim_attempts               58     0.003933
Pick'n'roll_roller_three_attempts             58     0.003933
Isolation_attempts                            56     0.003798
Isolation_makes                               56     0.003798
Isolation_rim_attempts                        56     0.003798
Isolation_three_attempts                      56     0.003798
Hand_offs_attempts                            52     0.003526
Hand_offs_makes                    

### Consistency check

I first removed a small number of exact duplicates and identifier-like fields. I also ran consistency checks to understand relationships among closely related columns (for example, cases where one variable was a sum of others).

In [ ]:
df["match_date"] = pd.to_datetime(df["match_date"], dayfirst=True, errors="coerce")

print("\n=== Date parsing check ===")
print("Missing parsed dates:", df["match_date"].isna().sum())
print("Min date:", df["match_date"].min())
print("Max date:", df["match_date"].max())

print("\n=== Logical consistency checks ===")

def mismatch_count(lhs_col, rhs_expr, name):
    bad = ~(np.isclose(lhs_col, rhs_expr, equal_nan=True))
    print(f"{name}: mismatched rows =", bad.sum())
    return bad.sum()

# REB = OREB + DREB
if all(c in df.columns for c in ["REB", "OREB", "DREB"]):
    mismatch_count(df["REB"], df["OREB"] + df["DREB"], "REB == OREB + DREB")

# FGM = 2FGM + 3FGM
if all(c in df.columns for c in ["FGM", "2FGM", "3FGM"]):
    mismatch_count(df["FGM"], df["2FGM"] + df["3FGM"], "FGM == 2FGM + 3FGM")

# FGA = 2FGA + 3FGA
if all(c in df.columns for c in ["FGA", "2FGA", "3FGA"]):
    mismatch_count(df["FGA"], df["2FGA"] + df["3FGA"], "FGA == 2FGA + 3FGA")

# team_points = 2*2FGM + 3*3FGM + FTM
if all(c in df.columns for c in ["team_points", "2FGM", "3FGM", "FTM"]):
    implied_points = 2 * df["2FGM"] + 3 * df["3FGM"] + df["FTM"]
    mismatch_count(df["team_points"], implied_points, "team_points from 2FGM/3FGM/FTM")

# total_points = team_points
if all(c in df.columns for c in ["team_points", "total_points"]):
    mismatch_count(df["team_points"], df["total_points"], "team_points == total_points")

# fg_points + ft_points = team_points
if all(c in df.columns for c in ["fg_points", "ft_points", "team_points"]):
    mismatch_count(df["team_points"], df["fg_points"] + df["ft_points"], "team_points == fg_points + ft_points")

# three_points = 3 * three_makes
if all(c in df.columns for c in ["three_points", "three_makes"]):
    mismatch_count(df["three_points"], 3 * df["three_makes"], "three_points == 3 * three_makes")

# above_break3_points = 3 * above_break3_makes
if all(c in df.columns for c in ["above_break3_points", "above_break3_makes"]):
    mismatch_count(df["above_break3_points"], 3 * df["above_break3_makes"], "above_break3_points == 3 * above_break3_makes")

# corner3_points = 3 * corner3_makes
if all(c in df.columns for c in ["corner3_points", "corner3_makes"]):
    mismatch_count(df["corner3_points"], 3 * df["corner3_makes"], "corner3_points == 3 * corner3_makes")

# ft_points = ft_makes
if all(c in df.columns for c in ["ft_points", "ft_makes"]):
    mismatch_count(df["ft_points"], df["ft_makes"], "ft_points == ft_makes")

print("\n=== Simple outlier scan (1st/99th percentiles) ===")

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

outlier_summary = []
for col in numeric_cols:
    s = df[col].dropna()
    if len(s) == 0:
        continue
    q01 = s.quantile(0.01)
    q99 = s.quantile(0.99)
    outlier_summary.append({
        "column": col,
        "min": s.min(),
        "q01": q01,
        "median": s.median(),
        "q99": q99,
        "max": s.max()
    })

outlier_df = pd.DataFrame(outlier_summary).sort_values("column")
print(outlier_df.head(50))




=== Date parsing check ===
Missing parsed dates: 0
Min date: 2020-12-23 03:00:00
Max date: 2026-03-05 06:30:00

=== Logical consistency checks ===
REB == OREB + DREB: mismatched rows = 12224
FGM == 2FGM + 3FGM: mismatched rows = 0
FGA == 2FGA + 3FGA: mismatched rows = 0
team_points from 2FGM/3FGM/FTM: mismatched rows = 12216
team_points == total_points: mismatched rows = 0
team_points == fg_points + ft_points: mismatched rows = 0
three_points == 3 * three_makes: mismatched rows = 0
above_break3_points == 3 * above_break3_makes: mismatched rows = 0
corner3_points == 3 * corner3_makes: mismatched rows = 0
ft_points == ft_makes: mismatched rows = 0

=== Simple outlier scan (1st/99th percentiles) ===
                               column        min        q01      median  \
18                               2FGA  11.000000  33.000000   51.000000   
17                               2FGM   4.000000  15.000000   26.000000   
19                            2FG_PCT   0.254545   0.348837    0.521

## Feature Engineering

Some raw features would leak the outcome because they are only recorded after the match. To avoid this, I converted them into rolling historical features so that only past team performance is used for prediction. For the first few games of a team, rolling means or standard deviations may be unavailable due to limited history, so these missing values were imputed later.

In [ ]:

# 0. Load step1-cleaned data
df["match_date"] = pd.to_datetime(df["match_date"], errors="coerce")
df = df.sort_values(["team_id", "match_date", "game_id"]).reset_index(drop=True)

print("Input shape:", df.shape)

# 1. Drop columns we already know are redundant / not useful

drop_now = [
    "_id",
    "total_points",   # duplicate of team_points
    "ft_makes",       # duplicate of FTM
    "ft_attempts",    # duplicate of FTA
    "our_fixture_id",
    "our_team_id",
]

df = df.drop(columns=[c for c in drop_now if c in df.columns]).copy()

print("After drop_now:", df.shape)


# 3. Create ratio / style features from raw same-game columns
#    (these raw versions still CANNOT go directly into final model;
#     we create them here so we can later make lagged/rolling versions)

eps = 1e-9

if set(["3FGA", "FGA"]).issubset(df.columns):
    df["three_pa_rate"] = df["3FGA"] / (df["FGA"] + eps)

if set(["FTA", "FGA"]).issubset(df.columns):
    df["fta_rate"] = df["FTA"] / (df["FGA"] + eps)

if set(["OREB", "REB"]).issubset(df.columns):
    df["oreb_share"] = df["OREB"] / (df["REB"] + eps)

if set(["assists", "FGM"]).issubset(df.columns):
    df["ast_per_fgm"] = df["assists"] / (df["FGM"] + eps)

if set(["turnovers", "possessions"]).issubset(df.columns):
    df["tov_per_poss"] = df["turnovers"] / (df["possessions"] + eps)

if set(["rim_attempts", "FGA"]).issubset(df.columns):
    df["rim_rate"] = df["rim_attempts"] / (df["FGA"] + eps)

if set(["corner3_attempts", "3FGA"]).issubset(df.columns):
    df["corner3_share"] = df["corner3_attempts"] / (df["3FGA"] + eps)

if set(["above_break3_attempts", "3FGA"]).issubset(df.columns):
    df["above_break3_share"] = df["above_break3_attempts"] / (df["3FGA"] + eps)

if set(["shot_distance_sum", "shot_distance_count"]).issubset(df.columns):
    df["avg_shot_distance"] = df["shot_distance_sum"] / (df["shot_distance_count"] + eps)


# 4. Rest / schedule features
df["prev_match_date"] = df.groupby("team_id")["match_date"].shift(1)
df["rest_days"] = (df["match_date"] - df["prev_match_date"]).dt.days

# first game for a team will be NaN; choose a neutral value later
df["is_b2b"] = (df["rest_days"] == 1).astype(float)

# games played in recent windows
def games_in_last_n_days(group, n_days):
    dates = group["match_date"]
    out = []
    for i, current_date in enumerate(dates):
        if i == 0:
            out.append(0)
        else:
            prev_dates = dates.iloc[:i]
            count = ((current_date - prev_dates).dt.days <= n_days).sum()
            out.append(count)
    return pd.Series(out, index=group.index)

df["games_last_7d"] = (
    df.groupby("team_id", group_keys=False)
      .apply(lambda g: games_in_last_n_days(g, 7))
)

# 5. Columns to roll
#    Start with a conservative set for baseline

roll_base_cols = [
    "team_points",
    "opponent_points",
    "ORTG",
    "DRTG",
    "possessions",
    "FG_PCT",
    "3FG_PCT",
    "FT_PCT",
    "three_pa_rate",
    "fta_rate",
]

roll_base_cols = [c for c in roll_base_cols if c in df.columns]

print("Rolling columns:")
print(roll_base_cols)

# 6. Helper functions for lagged rolling / expanding
def add_lagged_rolling_features(data, group_col, cols):
    data = data.copy()

    for col in cols:
        lagged = data.groupby(group_col)[col].shift(1)  # avoid leakage

        # recent form
        data[f"{col}_last5_mean"] = (
            lagged.groupby(data[group_col])
                  .transform(lambda s: s.rolling(5, min_periods=1).mean())
        )

        # slightly longer-term form
        data[f"{col}_last10_mean"] = (
            lagged.groupby(data[group_col])
                  .transform(lambda s: s.rolling(10, min_periods=1).mean())
        )

        # recent volatility
        data[f"{col}_last5_std"] = (
            lagged.groupby(data[group_col])
                  .transform(lambda s: s.rolling(5, min_periods=2).std())
        )

        # season-to-date level
        data[f"{col}_expanding_mean"] = (
            lagged.groupby(data[group_col])
                  .transform(lambda s: s.expanding(min_periods=1).mean())
        )

    return data


df = add_lagged_rolling_features(
    data=df,
    group_col="team_id",
    cols=roll_base_cols,

)

print("After rolling features:", df.shape)

# 7. Home/away-specific historical features

homeaway_cols = [
    "team_points",
    "opponent_points",
    "ORTG",
    "DRTG",
    "possessions"
]
homeaway_cols = [c for c in homeaway_cols if c in df.columns]

for col in homeaway_cols:
    lagged_col = df.groupby("team_id")[col].shift(1)

    # history only for games with same home/away status
    df[f"{col}_homeaway_expanding_mean"] = (
        lagged_col.groupby([df["team_id"], df["is_home"]])
                  .transform(lambda s: s.expanding(min_periods=1).mean())
    )


# 9. Fill only the schedule-related first-game missing values
if "rest_days" in df.columns:
    # first game has no previous game
    df["rest_days"] = df["rest_days"].fillna(7)

if "prev_match_date" in df.columns:
    df = df.drop(columns=["prev_match_date"])


Input shape: (14746, 125)
After drop_now: (14746, 119)


/var/folders/c1/gctr2p2503s1yq8n3gn7zg780000gn/T/ipykernel_51023/4287821838.py:79: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: games_in_last_n_days(g, 7))


Rolling columns:
['team_points', 'opponent_points', 'ORTG', 'DRTG', 'possessions', 'FG_PCT', '3FG_PCT', 'FT_PCT', 'three_pa_rate', 'fta_rate']
After rolling features: (14746, 172)


### Merge the table to one game one row

In [ ]:

# 1. post-game stats，can't be included in X
raw_current_game_cols = [
    "team_points",
    "opponent_points",
    "plus_minus",
    "ORTG",
    "DRTG",
    "possessions",
    "assists",
    "FGM", "FGA", "FG_PCT",
    "2FGM", "2FGA", "2FG_PCT",
    "3FGM", "3FGA", "3FG_PCT",
    "FTM", "FTA", "FT_PCT",
    "DREB", "OREB", "REB",
    "turnovers",
    "three_attempts", "three_makes", "three_points",
    "rim_attempts", "rim_makes", "rim_points",
    "short_mid_attempts", "short_mid_makes", "short_mid_points",
    "long_mid_attempts", "long_mid_makes", "long_mid_points",
    "above_break3_attempts", "above_break3_makes", "above_break3_points",
    "corner3_attempts", "corner3_makes", "corner3_points",
    "fg_points", "ft_points",

    "three_pa_rate",
    "fta_rate",
    "oreb_share",
    "ast_per_fgm",
    "tov_per_poss",
    "rim_rate",
    "corner3_share",
    "above_break3_share",
    "avg_shot_distance",
]

raw_current_game_cols = [c for c in raw_current_game_cols if c in df.columns]

# 2. merge
cols_for_merge = [
    c for c in df.columns
    if (c not in raw_current_game_cols) or (c in ["game_id", "match_date", "team_id", "team_name", "is_home", "team_points", "opponent_points"])
]

tmp = df[cols_for_merge].copy()

# 3. split home / away
home_df = tmp[tmp["is_home"] == 1].copy()
away_df = tmp[tmp["is_home"] == 0].copy()

# 4. rename
home_df = home_df.rename(columns={
    c: f"home_{c}" for c in home_df.columns if c not in ["game_id", "match_date"]
})
away_df = away_df.rename(columns={
    c: f"away_{c}" for c in away_df.columns if c not in ["game_id", "match_date"]
})

# 5. merge into one row per game
game_df = home_df.merge(
    away_df,
    on=["game_id", "match_date"],
    how="inner",
    validate="one_to_one"
)

# 6. final targets
game_df["home_points"] = game_df["home_team_points"]
game_df["away_points"] = game_df["away_team_points"]

# 7. drop duplicated target-source columns after creating y
game_df = game_df.drop(columns=[
    "home_team_points",
    "away_team_points",
    "home_opponent_points",
    "away_opponent_points",
])

# 8. a few matchup interaction features
if {"home_team_points_last5_mean", "away_opponent_points_last5_mean"}.issubset(game_df.columns):
    game_df["home_attack_vs_away_def_last5"] = (
        game_df["home_team_points_last5_mean"] - game_df["away_opponent_points_last5_mean"]
    )

if {"away_team_points_last5_mean", "home_opponent_points_last5_mean"}.issubset(game_df.columns):
    game_df["away_attack_vs_home_def_last5"] = (
        game_df["away_team_points_last5_mean"] - game_df["home_opponent_points_last5_mean"]
    )

if {"home_ORTG_last5_mean", "away_DRTG_last5_mean"}.issubset(game_df.columns):
    game_df["home_ortg_vs_away_drtg_last5"] = (
        game_df["home_ORTG_last5_mean"] - game_df["away_DRTG_last5_mean"]
    )

if {"away_ORTG_last5_mean", "home_DRTG_last5_mean"}.issubset(game_df.columns):
    game_df["away_ortg_vs_home_drtg_last5"] = (
        game_df["away_ORTG_last5_mean"] - game_df["home_DRTG_last5_mean"]
    )

if {"home_possessions_last5_mean", "away_possessions_last5_mean"}.issubset(game_df.columns):
    game_df["expected_pace_last5_sum"] = (
        game_df["home_possessions_last5_mean"] + game_df["away_possessions_last5_mean"]
    )

print(game_df.shape)
print(game_df[["game_id", "match_date", "home_team_name", "away_team_name", "home_points", "away_points"]].head())

(7373, 253)
   game_id          match_date       home_team_name      away_team_name  \
0   617993 2020-12-24 03:00:00  Cleveland Cavaliers   Charlotte Hornets   
1   618071 2020-12-28 03:30:00  Cleveland Cavaliers  Philadelphia 76ers   
2   619118 2020-12-30 03:00:00  Cleveland Cavaliers     New York Knicks   
3   618029 2021-01-12 03:00:00  Cleveland Cavaliers   Memphis Grizzlies   
4   618463 2021-01-13 03:30:00  Cleveland Cavaliers           Utah Jazz   

   home_points  away_points  
0          121          114  
1          118           94  
2           86           95  
3           91          101  
4           87          117  


## Compact feature selection

The full engineered feature space contained many overlapping variants of the same underlying concepts, including recent scoring level, defensive concession, pace, shooting efficiency, and shot mix. After rolling-window feature generation and home/away expansion, this redundancy increased substantially. Rather than retaining all such variants, I selected a compact set of representative features per concept. This improved out-of-sample performance relative to the larger feature set, suggesting that many of the additional variables added noise or low incremental signal.

In [ ]:
core_features = [
    "home_team_id", "away_team_id",

    "home_team_points_last5_mean",
    "home_team_points_last10_mean",
    "home_opponent_points_last5_mean",
    "home_opponent_points_last10_mean",
    "away_team_points_last5_mean",
    "away_team_points_last10_mean",
    "away_opponent_points_last5_mean",
    "away_opponent_points_last10_mean",

    "home_ORTG_last5_mean",
    "home_DRTG_last5_mean",
    "home_possessions_last5_mean",
    "away_ORTG_last5_mean",
    "away_DRTG_last5_mean",
    "away_possessions_last5_mean",

    "home_FG_PCT_last5_mean",
    "home_3FG_PCT_last5_mean",
    "home_FT_PCT_last5_mean",
    "home_three_pa_rate_last5_mean",
    "home_fta_rate_last5_mean",
    "away_FG_PCT_last5_mean",
    "away_3FG_PCT_last5_mean",
    "away_FT_PCT_last5_mean",
    "away_three_pa_rate_last5_mean",
    "away_fta_rate_last5_mean",

    "home_rest_days",
    "home_is_b2b",
    "home_games_last_7d",
    "away_rest_days",
    "away_is_b2b",
    "away_games_last_7d",

    "home_attack_vs_away_def_last5",
    "away_attack_vs_home_def_last5",
    "home_ortg_vs_away_drtg_last5",
    "away_ortg_vs_home_drtg_last5",
    "expected_pace_last5_sum",

    "home_team_points_homeaway_expanding_mean",
    "home_opponent_points_homeaway_expanding_mean",
    "away_team_points_homeaway_expanding_mean",
    "away_opponent_points_homeaway_expanding_mean",
]

core_features = [c for c in core_features if c in game_df.columns]


## Split train and test

In [ ]:
# 1. sort by date
game_df = game_df.sort_values(["match_date", "game_id"]).reset_index(drop=True)

# 2. train / val / test
n = len(game_df)
train_end = int(n * 0.7)
val_end = int(n * 0.85)

train_df = game_df.iloc[:train_end].copy()
val_df = game_df.iloc[train_end:val_end].copy()
test_df = game_df.iloc[val_end:].copy()

print("train shape:", train_df.shape)
print("val shape:", val_df.shape)
print("test shape:", test_df.shape)

print("train date range:", train_df["match_date"].min(), "->", train_df["match_date"].max())
print("val date range:", val_df["match_date"].min(), "->", val_df["match_date"].max())
print("test date range:", test_df["match_date"].min(), "->", test_df["match_date"].max())

feature_cols = core_features.copy()

X_train = train_df[feature_cols].copy()
X_val = val_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()

y_train_home = train_df["home_points"].copy()
y_val_home = val_df["home_points"].copy()
y_test_home = test_df["home_points"].copy()

y_train_away = train_df["away_points"].copy()
y_val_away = val_df["away_points"].copy()
y_test_away = test_df["away_points"].copy()

print("num features:", len(feature_cols))
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

train shape: (5161, 253)
val shape: (1106, 253)
test shape: (1106, 253)
train date range: 2020-12-23 03:00:00 -> 2024-10-27 03:00:00
val date range: 2024-10-27 03:00:00 -> 2025-04-03 03:00:00
test date range: 2025-04-03 03:30:00 -> 2026-03-05 06:30:00
num features: 41
X_train: (5161, 41)
X_val: (1106, 41)
X_test: (1106, 41)


### Imputation

In [ ]:

std_cols = [c for c in feature_cols if c.endswith("_std")]

X_train_imp = X_train.copy()
X_val_imp = X_val.copy()
X_test_imp = X_test.copy()

for col in std_cols:
    X_train_imp[col] = X_train_imp[col].fillna(0.0)
    X_val_imp[col] = X_val_imp[col].fillna(0.0)
    X_test_imp[col] = X_test_imp[col].fillna(0.0)

# use train fit mean imputer
other_cols = [c for c in feature_cols if c not in std_cols]

imputer = SimpleImputer(strategy="mean")
X_train_imp[other_cols] = imputer.fit_transform(X_train_imp[other_cols])
X_val_imp[other_cols] = imputer.transform(X_val_imp[other_cols])
X_test_imp[other_cols] = imputer.transform(X_test_imp[other_cols])

print("remaining missing in X_train:", X_train_imp.isna().sum().sum())
print("remaining missing in X_val:", X_val_imp.isna().sum().sum())
print("remaining missing in X_test:", X_test_imp.isna().sum().sum())

remaining missing in X_train: 0
remaining missing in X_val: 0
remaining missing in X_test: 0


### Mean Model

In [ ]:
home_mean_model = HistGradientBoostingRegressor(
    loss="squared_error",
    learning_rate=0.05,
    max_depth=4,
    max_iter=300,
    min_samples_leaf=20,
    random_state=42
)

away_mean_model = HistGradientBoostingRegressor(
    loss="squared_error",
    learning_rate=0.05,
    max_depth=4,
    max_iter=300,
    min_samples_leaf=20,
    random_state=42
)

# train fit
home_mean_model.fit(X_train_imp, y_train_home)
away_mean_model.fit(X_train_imp, y_train_away)

# train / val / test predictions from mean models
mu_train_home = home_mean_model.predict(X_train_imp)
mu_val_home = home_mean_model.predict(X_val_imp)
mu_test_home = home_mean_model.predict(X_test_imp)

mu_train_away = away_mean_model.predict(X_train_imp)
mu_val_away = away_mean_model.predict(X_val_imp)
mu_test_away = away_mean_model.predict(X_test_imp)

### Use validation set to learn sigma

In [ ]:
# use validation residuals to estimate base predictive sigma
home_resid_val = y_val_home - mu_val_home
away_resid_val = y_val_away - mu_val_away

sigma_home_base = np.std(home_resid_val, ddof=1)
sigma_away_base = np.std(away_resid_val, ddof=1)

print("sigma_home_base:", sigma_home_base)
print("sigma_away_base:", sigma_away_base)

sigma_home_base: 12.063067139138692
sigma_away_base: 12.14647061608096


### Calibration scaling

In [ ]:
results = []

for c in [1.0, 1.25, 1.5, 1.75, 2.0, 2.5, 3.0]:
    sigma_home_val_scaled = np.full(len(X_val_imp), sigma_home_base * c, dtype=float)
    sigma_away_val_scaled = np.full(len(X_val_imp), sigma_away_base * c, dtype=float)

    home_nll = gaussian_nll(y_val_home.values, mu_val_home, sigma_home_val_scaled)
    away_nll = gaussian_nll(y_val_away.values, mu_val_away, sigma_away_val_scaled)

    home_cov, home_width = interval_coverage(y_val_home.values, mu_val_home, sigma_home_val_scaled)
    away_cov, away_width = interval_coverage(y_val_away.values, mu_val_away, sigma_away_val_scaled)

    results.append({
        "scale": c,
        "home_nll": home_nll,
        "home_cov80": home_cov,
        "home_width80": home_width,
        "away_nll": away_nll,
        "away_cov80": away_cov,
        "away_width80": away_width,
    })

results_df = pd.DataFrame(results)
results_df

NameError: name 'gaussian_nll' is not defined

### Build Evaluation Metrics

In [ ]:
def gaussian_nll(y_true, mu, sigma):
    sigma = np.clip(sigma, 1e-6, None)
    return np.mean(0.5 * np.log(2 * np.pi * sigma**2) + 0.5 * ((y_true - mu) / sigma)**2)

def interval_coverage(y_true, mu, sigma, z=1.2815515655446004):
    lower = mu - z * sigma
    upper = mu + z * sigma
    covered = ((y_true >= lower) & (y_true <= upper)).mean()
    avg_width = np.mean(upper - lower)
    return covered, avg_width

def evaluate_predictions(y_true, mu, sigma, label="target"):
    mae = mean_absolute_error(y_true, mu)
    rmse = np.sqrt(mean_squared_error(y_true, mu))
    nll = gaussian_nll(y_true, mu, sigma)
    coverage_80, width_80 = interval_coverage(y_true, mu, sigma)

    print(f"\n=== {label} ===")
    print("MAE:", round(mae, 4))
    print("RMSE:", round(rmse, 4))
    print("Gaussian NLL:", round(nll, 4))
    print("80% interval coverage:", round(coverage_80, 4))
    print("80% avg interval width:", round(width_80, 4))



### Retrain the mean model using train+validation

In [ ]:
# combine train + val for final mean-model training
trainval_df = pd.concat([train_df, val_df], axis=0).sort_values(["match_date", "game_id"]).reset_index(drop=True)

X_trainval = trainval_df[feature_cols].copy()
y_trainval_home = trainval_df["home_points"].copy()
y_trainval_away = trainval_df["away_points"].copy()

# re-do imputation using train+val fit
std_cols = [c for c in feature_cols if c.endswith("_std")]

X_trainval_imp = X_trainval.copy()
X_test_final_imp = X_test.copy()

for col in std_cols:
    X_trainval_imp[col] = X_trainval_imp[col].fillna(0.0)
    X_test_final_imp[col] = X_test_final_imp[col].fillna(0.0)

other_cols = [c for c in feature_cols if c not in std_cols]

imputer_final = SimpleImputer(strategy="mean")
X_trainval_imp[other_cols] = imputer_final.fit_transform(X_trainval_imp[other_cols])
X_test_final_imp[other_cols] = imputer_final.transform(X_test_final_imp[other_cols])

# refit final mean models on train+val
home_mean_model_final = HistGradientBoostingRegressor(
    loss="squared_error",
    learning_rate=0.05,
    max_depth=4,
    max_iter=300,
    min_samples_leaf=20,
    random_state=42
)

away_mean_model_final = HistGradientBoostingRegressor(
    loss="squared_error",
    learning_rate=0.05,
    max_depth=4,
    max_iter=300,
    min_samples_leaf=20,
    random_state=42
)

home_mean_model_final.fit(X_trainval_imp, y_trainval_home)
away_mean_model_final.fit(X_trainval_imp, y_trainval_away)

mu_test_home_final = home_mean_model_final.predict(X_test_final_imp)
mu_test_away_final = away_mean_model_final.predict(X_test_final_imp)

# final sigma: validation-based, no extra scaling
sigma_test_home_final = np.full(len(X_test_final_imp), sigma_home_base, dtype=float)
sigma_test_away_final = np.full(len(X_test_final_imp), sigma_away_base, dtype=float)

evaluate_predictions(y_test_home.values, mu_test_home_final, sigma_test_home_final, "home_points_final")
evaluate_predictions(y_test_away.values, mu_test_away_final, sigma_test_away_final, "away_points_final")


=== home_points_final ===
MAE: 10.2291
RMSE: 13.0963
Gaussian NLL: 3.9984
80% interval coverage: 0.7776
80% avg interval width: 30.9189

=== away_points_final ===
MAE: 9.7659
RMSE: 12.6027
Gaussian NLL: 3.9542
80% interval coverage: 0.7929
80% avg interval width: 31.1327


## Model Specification

### Output example

In [ ]:
z80 = 1.2815515655446004  # central 80% interval for Normal

pred_df = test_df[["game_id", "match_date", "home_team_name", "away_team_name"]].copy()

pred_df["home_points_actual"] = y_test_home.values
pred_df["away_points_actual"] = y_test_away.values

pred_df["home_mu"] = mu_test_home_final
pred_df["home_sigma"] = sigma_test_home_final
pred_df["home_p10"] = pred_df["home_mu"] - z80 * pred_df["home_sigma"]
pred_df["home_p90"] = pred_df["home_mu"] + z80 * pred_df["home_sigma"]

pred_df["away_mu"] = mu_test_away_final
pred_df["away_sigma"] = sigma_test_away_final
pred_df["away_p10"] = pred_df["away_mu"] - z80 * pred_df["away_sigma"]
pred_df["away_p90"] = pred_df["away_mu"] + z80 * pred_df["away_sigma"]

pred_df.head()

,game_id,match_date,home_team_name,away_team_name,home_points_actual,away_points_actual,home_mu,home_sigma,home_p10,home_p90,away_mu,away_sigma,away_p10,away_p90
6267,2168775,2025-04-03 03:30:00,Dallas Mavericks,Atlanta Hawks,120,118,121.402327,12.063067,105.942884,136.861769,119.369031,12.146471,103.802702,134.935359
6268,2168776,2025-04-03 04:00:00,Denver Nuggets,San Antonio Spurs,106,113,123.147927,12.063067,107.688485,138.607370,114.090902,12.146471,98.524574,129.657231
6269,2168777,2025-04-03 05:00:00,Los Angeles Clippers,New Orleans Pelicans,114,98,115.592533,12.063067,100.133091,131.051976,104.788195,12.146471,89.221866,120.354523
6270,2168778,2025-04-04 02:00:00,Washington Wizards,Orlando Magic,97,109,105.521258,12.063067,90.061815,120.980700,117.093340,12.146471,101.527012,132.659669
6271,2168781,2025-04-04 02:00:00,Philadelphia 76ers,Milwaukee Bucks,113,126,112.092376,12.063067,96.632933,127.551818,114.638128,12.146471,99.071800,130.204457


### Pick one game to demonstrate

In [ ]:
example = pred_df.iloc[0]

print("Match:")
print(example["home_team_name"], "vs", example["away_team_name"])
print("Date:", example["match_date"])
print()

print("Home team predictive distribution:")
print(f"{example['home_team_name']} ~ N({example['home_mu']:.2f}, {example['home_sigma']:.2f}^2)")
print(f"Point prediction (mean): {example['home_mu']:.2f}")
print(f"80% prediction interval: [{example['home_p10']:.2f}, {example['home_p90']:.2f}]")
print(f"Actual points: {example['home_points_actual']}")
print()

print("Away team predictive distribution:")
print(f"{example['away_team_name']} ~ N({example['away_mu']:.2f}, {example['away_sigma']:.2f}^2)")
print(f"Point prediction (mean): {example['away_mu']:.2f}")
print(f"80% prediction interval: [{example['away_p10']:.2f}, {example['away_p90']:.2f}]")
print(f"Actual points: {example['away_points_actual']}")

Match:
Dallas Mavericks vs Atlanta Hawks
Date: 2025-04-03 03:30:00

Home team predictive distribution:
Dallas Mavericks ~ N(121.40, 12.06^2)
Point prediction (mean): 121.40
80% prediction interval: [105.94, 136.86]
Actual points: 120

Away team predictive distribution:
Atlanta Hawks ~ N(119.37, 12.15^2)
Point prediction (mean): 119.37
80% prediction interval: [103.80, 134.94]
Actual points: 118
